# Categorical raster reclassification

Recode a detailed categorical raster into broader analytical classes while preserving integer categories and explicit NoData handling.


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import rasterio

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == 'notebooks' else HERE
DATA = ROOT / 'data_raw'
OUTPUTS = ROOT / 'outputs'
TABLES = ROOT / 'tables'
for folder in (OUTPUTS, TABLES):
    folder.mkdir(parents=True, exist_ok=True)

CLASS_RASTER = DATA / 'settlement_or_landcover.tif'
OUTPUT_RASTER = OUTPUTS / 'categorical_raster_reclassified.tif'
CLASS_MAPPING = {}
CLASS_LABELS = {}
NODATA_OUT = -9999

if not CLASS_RASTER.exists():
    raise FileNotFoundError(f'Categorical raster not found: {CLASS_RASTER}')
if not CLASS_MAPPING:
    raise ValueError('CLASS_MAPPING is empty. Add the input-code to output-class mapping before running.')

with rasterio.open(CLASS_RASTER) as src:
    arr = src.read(1, masked=False)
    profile = src.profile.copy()
    nodata_in = src.nodata

out = np.full(arr.shape, NODATA_OUT, dtype='int16')
for old_value, new_value in CLASS_MAPPING.items():
    out[arr == old_value] = int(new_value)
if nodata_in is not None:
    out[arr == nodata_in] = NODATA_OUT

profile.update(dtype='int16', nodata=NODATA_OUT, compress='lzw')
with rasterio.open(OUTPUT_RASTER, 'w', **profile) as dst:
    dst.write(out, 1)

valid = out[out != NODATA_OUT]
counts = pd.Series(valid).value_counts().sort_index().rename_axis('class_code').reset_index(name='cell_count')
counts['class_label'] = counts['class_code'].map(CLASS_LABELS).fillna('unlabelled')
counts.to_csv(TABLES / 'categorical_reclassification_counts.csv', index=False)
with open(OUTPUTS / 'categorical_reclassification_report.json', 'w', encoding='utf-8') as f:
    json.dump({'run_time_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'), 'input': str(CLASS_RASTER.relative_to(ROOT)), 'output': str(OUTPUT_RASTER.relative_to(ROOT)), 'nodata_out': NODATA_OUT, 'mapping': CLASS_MAPPING, 'labels': CLASS_LABELS}, f, indent=2)
counts
